# Paper 3 — Corrected Pipeline (Colab runner)

Runs the audit-corrected pipeline end to end and produces every report.

**Before you start:** set the runtime to GPU (*Runtime → Change runtime type → T4 GPU*).
The pipeline runs fine on CPU, but the neural training is the bulk of the wall time.

**What you need to upload** (two things):
1. the `code/` directory
2. `Paper3_MegaDataset_SPEI_FINAL.csv`

They must end up laid out as:
```
PROJECT_ROOT/
├── Paper3_MegaDataset_SPEI_FINAL.csv
└── code/
```
`config.py` derives `PROJECT_ROOT` from `code/`'s parent and writes everything to
`PROJECT_ROOT/outputs/`, so this layout matters.

## 1. Dependencies

Colab already ships numpy, pandas, scikit-learn, scipy, matplotlib, seaborn, torch, lightgbm, xgboost and statsmodels. These are the ones it does not.

In [ ]:
!pip -q install catboost shap optuna

# Optional: only used for the choropleth maps. Every call site is wrapped in
# try/except, so skipping this just means no maps -- no other output changes.
!pip -q install geopandas || echo "geopandas unavailable; choropleth maps will be skipped"

## 2. Get the project into place

Pick **one** of the two options below.

In [ ]:
# --- OPTION A: Google Drive (recommended -- survives disconnects) ---
from google.colab import drive
drive.mount('/content/drive')

# Point this at the folder that contains BOTH code/ and the CSV.
PROJECT_ROOT = '/content/drive/MyDrive/Paper3_MERGED'

import os
print('exists:', os.path.isdir(PROJECT_ROOT))
print(sorted(os.listdir(PROJECT_ROOT))[:10])

In [ ]:
# --- OPTION B: direct upload (zip code/ + the CSV together first) ---
# from google.colab import files
# up = files.upload()                      # choose Paper3_MERGED.zip
# !unzip -q -o Paper3_MERGED.zip -d /content/
# PROJECT_ROOT = '/content/Paper3_MERGED'

## 3. Pre-flight checks

Fails loudly here rather than 40 minutes into the run.

In [ ]:
import os, sys, torch

CODE_DIR = os.path.join(PROJECT_ROOT, 'code')
CSV = os.path.join(PROJECT_ROOT, 'Paper3_MegaDataset_SPEI_FINAL.csv')

assert os.path.isdir(CODE_DIR), f'code/ not found at {CODE_DIR}'
assert os.path.exists(CSV), (
    f'dataset not found at {CSV} -- it must sit NEXT TO code/, not inside it')

print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Running on CPU -- expect roughly 1.5-2h.')

print('dataset size: %.1f MB' % (os.path.getsize(CSV) / 1e6))

os.chdir(CODE_DIR)
sys.path.insert(0, CODE_DIR)
print('cwd:', os.getcwd())

## 4. Protocol compliance tests

Run these first — they take well under a minute and check the invariants the audit
found violated (six LOSO states, zero partition overlap, FIT-only fitting, genuine
joint-vs-post-hoc divergence, correct Nemenyi `k`). If any fail, stop: the pipeline
results would not be trustworthy.

In [ ]:
!python -m unittest test_protocol_compliance test_pipeline 2>&1 | tail -20

## 5. Run the full pipeline

Streams the log live. Expect roughly:

| Stage | Notes |
|---|---|
| Feature engineering + selection | 95 → 83 → 73 → 74 features, all fitted on FIT only |
| Temporal train / refit / calibration | FIT 1985–2013, DEV 2014–2015, CAL 2016–2018, TEST 2019–2023 |
| SHAP across 4 backbones | real explainers, no synthetic fallback |
| O4 joint vs post-hoc | two full trainings under matched conditions |
| LOSO | 6 folds, each with its own feature selection |
| Ablation | 5 configurations + distinctness checks |
| Validator + claim audit + final audit | property-based, not hard-coded |

In [ ]:
import subprocess, sys

proc = subprocess.Popen([sys.executable, '-u', 'main.py'],
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1)

KEY = ('FEATURE FUNNEL', 'LEAKAGE AUDIT', 'FROZEN ENSEMBLE', 'SHAP consistency',
       'LOSO Fold', '-> RMSE=', 'O4 arm', 'Objective O', 'Ablation config',
       'Ablation distinctness', 'Methodology validation', 'Claim consistency',
       'Final methodology audit', 'Traceback', 'Error', 'ERROR')

for line in proc.stdout:
    if any(k in line for k in KEY):
        print(line.rstrip())

rc = proc.wait()
print('\n=== pipeline exited with code', rc, '===')

## 6. Headline results

Read straight out of the generated artefacts.

In [ ]:
import json, os
import pandas as pd

R = os.path.join(PROJECT_ROOT, 'outputs', 'reports')

def load(name):
    p = os.path.join(R, name)
    return json.load(open(p)) if os.path.exists(p) else None

val, leak = load('methodology_validation_report.json'), load('leakage_provenance_audit.json')
claims = load('claim_consistency_audit.json')

if val:
    print('Methodology validation:', val['overall_status'], val['counts'])
    for c in val['checks']:
        if c['status'] != 'PASS':
            print(f"  {c['status']:8s} {c['id']}: {c['requirement']}")
            print(f"           -> {c['evidence'][:160]}")

if leak:
    print('\nLeakage audit:', 'PASS' if leak['all_passed'] else 'FAIL',
          f"({leak['n_experiments']} experiments)")
    print(pd.DataFrame([{
        'experiment': e['experiment'],
        'train/test': e.get('train_test_overlap'),
        'val/test': e.get('validation_test_overlap'),
        'cal/test': e.get('calibration_test_overlap'),
        'train/val': e.get('train_validation_overlap'),
        'status': e['status'],
    } for e in leak['experiments']]).to_string(index=False))

if claims:
    print('\nClaim consistency:', claims['status'],
          f"({claims['n_unsupported']} unsupported / {claims['n_matches']} matches)")

In [ ]:
# LOSO, ablation, and the four objectives
loso_p = os.path.join(R, 'loso_fold_metrics.csv')
if os.path.exists(loso_p):
    df = pd.read_csv(loso_p)
    cols = [c for c in ['state','rmse','mae','r_squared','picp','mpiw','ace',
                        'winkler_score','neuralcqr_ensemble_weight','n_features']
            if c in df.columns]
    print('=== LOSO (6 folds) ===')
    print(df[cols].to_string(index=False))
    num = [c for c in ['rmse','mae','r_squared','picp','mpiw'] if c in df.columns]
    print('\nmacro mean:', {c: round(df[c].mean(), 4) for c in num})

abl = load('ablation_report.json')
if abl:
    print('\n=== Ablation ===')
    keep = ['config','training_paradigm','n_optimization_stages','calibration',
            'rmse','mae','r_squared','picp','mpiw','ace','winkler_score']
    print(pd.DataFrame([{k: c.get(k) for k in keep if k in c}
                        for c in abl['configurations']]).to_string(index=False))
    print('all configurations distinct:', abl.get('all_configurations_distinct'))

o4, o5, o6 = load('objective_O4_report.json'), load('objective_o5_report.json'), load('objective_O6_report.json')
if o4:
    print('\n=== O4 ===', o4['verdict'])
    print('  point improvement supported   :', o4['point_prediction_improvement_supported'])
    print('  interval improvement supported:', o4['interval_quality_improvement_supported'])
if o5:
    pi = o5['primary_inference']
    print('\n=== O5 === slope', pi['slope'], 'CI', pi['ci_95'], 'p', pi['p_value'],
          '| R2', o5['variance_explained_fraction'])
    print('  magnitude:', o5['effect_magnitude_label'],
          '| detectable:', o5['statistically_detectable'],
          '| effective n:', o5['effective_sample_size']['n_effective'])
if o6:
    fr = o6['friedman_test']
    print('\n=== O6 === k =', o6['n_methods_compared'], '|', o6['ranking_status'])
    print('  friedman executed:', fr.get('test_executed'), '| p =', fr.get('p_value'))
    print('  claims statistical superiority:', o6['claims_statistical_superiority'])

## 7. Save results back to Drive

Skip if you already ran from Drive (`outputs/` is written in place).

In [ ]:
import shutil, datetime
stamp = datetime.datetime.now().strftime('%Y%m%d_%H%M')
archive = f'/content/paper3_outputs_{stamp}'
shutil.make_archive(archive, 'zip', os.path.join(PROJECT_ROOT, 'outputs'))
print('wrote', archive + '.zip')

# from google.colab import files
# files.download(archive + '.zip')